## MosKita — YOLOv8 Training
### Dengue Mosquito Breeding Site Detection

This notebook trains a YOLOv8s object detection model to identify *Aedes aegypti* / *Aedes albopictus* breeding containers.
The dataset is annotated in Roboflow and exported in YOLOv8 format with **8 classes (V1)**.

**Hardware:** Lenovo Legion 5 — RTX 2060 6 GB, Ryzen 7 4800H, 16 GB RAM

**Workflow:**
1. Environment & GPU check
2. Configuration ← *edit here before running*
3. Dataset assembly (merge selected sources)
4. Dataset verification & visualisation
5. Model training
6. Results & metrics
7. Model export (ONNX / TFLite)
8. Quick inference test
9. Summary & next steps


---
### 1 · Environment Setup

In [ ]:
import os
import sys
import time
import random
import yaml
import glob

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter

# Attempt to import ultralytics and YOLO; install if missing
try:
    from ultralytics import YOLO
    import ultralytics
except ModuleNotFoundError:
    print("\u26a0\ufe0f  'ultralytics' package not found. Installing via pip...")
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    except Exception as e:
        print(f"\u26a0\ufe0f  Failed to install ultralytics: {e}")
        raise
    # Retry import after installation
    from ultralytics import YOLO
    import ultralytics

print("\u2705 All libraries imported successfully!")
print(f"   PyTorch version      : {torch.__version__}")
print(f"   Ultralytics version  : {ultralytics.__version__}")
print(f"   CUDA available       : {torch.cuda.is_available()}")
print(f"   Device count         : {torch.cuda.device_count()}")
print(f"   Python               : {sys.version.split()[0]}")


#### Detect GPU Available, Details, CUDA, and cuDNN

In [ ]:
print("=" * 50)
print("\U0001f5a5\ufe0f  GPU INFORMATION")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_cap  = torch.cuda.get_device_capability(0)
    print(f"\u2705 GPU Detected         : {gpu_name}")
    print(f"   \u2022 Compute Capability : {gpu_cap[0]}.{gpu_cap[1]}")
    print(f"   \u2022 Total VRAM         : {gpu_mem:.2f} GB")
    print(f"   \u2022 VRAM Allocated     : {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    print(f"   \u2022 VRAM Reserved      : {torch.cuda.memory_reserved(0) / (1024**3):.2f} GB")
    DEVICE = 0
    # With fixed input size (640×640), benchmark mode selects the fastest cuDNN kernels
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False  # determinism handled via SEED in config
else:
    print("\u26a0\ufe0f  No GPU detected — training will use CPU (very slow)")
    DEVICE = "cpu"

print("=" * 50)
print()
print("=" * 50)
print("\u26a1 CUDA / PYTORCH INFORMATION")
print("=" * 50)
print(f"\u2705 CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   \u2022 PyTorch CUDA Ver.  : {torch.version.cuda}")
    print(f"   \u2022 cuDNN Benchmark    : {torch.backends.cudnn.benchmark}")
print(f"   \u2022 PyTorch Version    : {torch.__version__}")
print(f"\u2705 cuDNN Version        : {torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else 'N/A'}")
print("=" * 50)


---
### 2 · Configuration

Edit the cell below — dataset sources, model variant, and all hyperparameters — then run it before proceeding.


In [ ]:

# ================================================================
# ⚙️  CONFIGURATION  ←  Only this cell needs editing
# ================================================================

# ── DATASET SOURCE SELECTION ─────────────────────────────────
# Set enabled=True / False to include or exclude a source.
# After changing selections, re-run Section 3 (Dataset Assembly)
# to rebuild the training directory before training.
#
# Notes:
#   • adnans_breeding — mixed box/polygon export; polygon rows auto-converted.
#   • adnans_water  — polygon segmentation; auto-converted to boxes.
#   • faiyaz_mosquitofusion — mosquito/swarm classes already removed;
#     only "Breeding Place" (→ uncovered_container) remains.
#     Polygon rows are auto-converted to boxes.
#   • k1taru_local — self-curated Roboflow export with polygon labels;
#     polygons are auto-converted to boxes. Train split already has
#     4× photometric augmentation baked in.

DATASET_SOURCES = {
    "adnans_breeding": {
        "enabled":    True,
        "input_root": "../data/annotated/outsource/adnans/Breeding Place Detection",
        "class_map":  "../scripts/class_maps/adnans_breeding_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--convert-segments-to-boxes"],
        "note":       "Adnans — tires, drain inlets, vases, bottles (→ 4 V1 classes)",
    },
    "adnans_water": {
        "enabled":    False,
        "input_root": "../data/annotated/outsource/adnans/Water Surface Segmentation",
        "class_map":  "../scripts/class_maps/adnans_water_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--convert-segments-to-boxes"],
        "note":       "Adnans — water surface segmentation (polygon → box conversion)",
    },
    "faiyaz_mosquitofusion": {
        "enabled":    True,
        "input_root": "../data/annotated/outsource/faiyazabdullah/MosquitoFusion Dataset",
        "class_map":  "../scripts/class_maps/faiyaz_mosquitofusion_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--drop-unmapped", "--convert-segments-to-boxes"],
        "note":       "Faiyaz — MosquitoFusion (breeding place only)",
    },
    "roboflow_public": {
        "enabled":    True,
        "input_root": "../data/annotated/outsource/roboflow",
        "class_map":  "../scripts/class_maps/roboflow_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": [],
        "note":       "Roboflow — public breeding site dataset (tire, bucket, puddle)",
    },
    "k1taru_local": {
        "enabled":    True,
        "input_root": "../data/annotated/k1taru",
        "class_map":  "../scripts/class_maps/k1taru_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--convert-segments-to-boxes"],
        "note":       "K1taru — self-curated export (basin→uncovered_container, drum→drum)",
    },
}

# ── PATHS ─────────────────────────────────────────────────────
DATA_YAML     = "../data/annotated/data.yaml"
PROJECT_DIR   = "../models/runs"
EXPORT_DIR    = "../models/exports"
ASSEMBLED_DIR = "../data/annotated"      # target for assembly output

TARGET_NAMES = (
    "discarded_tire,flower_pot,bottle,coconut_shell,basin,"
    "uncovered_container,drain_inlet,stagnant_puddle,drum,bucket,"
    "styrofoam_container"
)

# ── MODEL ─────────────────────────────────────────────────────
MODEL_VARIANT = "yolov8s.pt"   # yolov8n | yolov8s | yolov8m | yolov8l | yolov8x
RUN_NAME      = "moskita_v1"
EXIST_OK      = False          # True = overwrite an existing run with the same name

# ── TRAINING HYPERPARAMETERS ──────────────────────────────────
EPOCHS          = 100           # max training epochs
BATCH_SIZE      = 24            # fits RTX 2060 6 GB VRAM
IMG_SIZE        = 640           # YOLOv8 default input resolution
PATIENCE        = 15            # early-stopping patience (epochs without improvement)
OPTIMIZER       = "AdamW"
LR0             = 0.001         # initial learning rate
LRF             = 0.01          # final LR as a fraction of lr0
WEIGHT_DECAY    = 0.0005        # L2 regularisation
WARMUP_EPOCHS   = 3.0           # linear warmup duration
WARMUP_MOMENTUM = 0.8           # initial momentum during warmup
CLOSE_MOSAIC    = 10            # disable mosaic for last N epochs (aids convergence)
WORKERS         = 4             # DataLoader worker threads
SAVE_PERIOD     = -1            # checkpoint every N epochs (-1 = disabled)

# ── AUGMENTATION ──────────────────────────────────────────────
MOSAIC      = 1.0   # mosaic aug probability
MIXUP       = 0.0   # mixup aug probability
COPY_PASTE  = 0.0   # copy-paste aug probability (raise to 0.2–0.3 for imbalanced classes)
FLIPUD      = 0.0   # vertical flip probability
FLIPLR      = 0.5   # horizontal flip probability
HSV_H       = 0.015 # hue shift range
HSV_S       = 0.7   # saturation shift range
HSV_V       = 0.4   # value (brightness) shift range
DEGREES     = 10.0  # rotation range (degrees)
TRANSLATE   = 0.1   # translation range (fraction of image)
SCALE       = 0.5   # scale range (fraction)
SHEAR       = 2.0   # shear range (degrees)
PERSPECTIVE = 0.0   # perspective distortion strength

# ── INFERENCE  (used at export / prediction time only) ────────
CONF_THRESHOLD = 0.4    # minimum detection confidence
IOU_THRESHOLD  = 0.45   # NMS IoU threshold

# ── MISC ──────────────────────────────────────────────────────
CACHE = False   # cache images in RAM — set True only if ≥32 GB RAM
SEED  = 42

# ── SEED INIT ─────────────────────────────────────────────────
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── SUMMARY ───────────────────────────────────────────────────
_enabled = [k for k, v in DATASET_SOURCES.items() if     v["enabled"]]
_skipped = [k for k, v in DATASET_SOURCES.items() if not v["enabled"]]

print("✅ Configuration loaded")
print(f"   Model       : {MODEL_VARIANT}  →  run '{RUN_NAME}'")
print(f"   Image size  : {IMG_SIZE}×{IMG_SIZE}    Batch: {BATCH_SIZE}    Epochs: {EPOCHS}  (patience={PATIENCE})")
print(f"   Optimizer   : {OPTIMIZER}  lr0={LR0}  lrf={LRF}  wd={WEIGHT_DECAY}")
print(f"   Sources ON  : {', '.join(_enabled) if _enabled else '(none)'}")
print(f"   Sources OFF : {', '.join(_skipped) if _skipped else '(none)'}")


---
### 3 · Dataset Assembly

Toggle `enabled` flags in the **Configuration** cell, then run the cell below to clear and rebuild the training directory from the selected sources. Re-run any time the source selection changes.

> ⚠️ This overwrites `data/annotated/train|val|test/`. Source datasets must have class IDs compatible with V1 (handled automatically by the remap script + class maps in `scripts/class_maps/`).


In [ ]:

# Remap and merge selected sources into ASSEMBLED_DIR.
# Calls scripts/remap_yolo_dataset.py for each enabled source.
# ⚠️  This CLEARS data/annotated/train|val|test/ before rebuilding.

import os
import shutil
import subprocess
import sys
from pathlib import Path

_nb_dir    = Path(os.getcwd()).resolve()   # notebooks/
_root_dir  = _nb_dir.parent               # MosKita/
_remap_py  = _root_dir / "scripts" / "remap_yolo_dataset.py"
_assembled = (_nb_dir / ASSEMBLED_DIR).resolve()
_SPLITS    = ["train", "val", "test"]

# ── Clear existing splits ──────────────────────────────────────
print("🗑️  Clearing assembled dataset directories...")
for split in _SPLITS:
    for sub in ("images", "labels"):
        d = _assembled / split / sub
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

# Clear the assembled YAML too so target-name renames rebuild cleanly.
_output_yaml = _assembled / "data.yaml"
if _output_yaml.exists():
    _output_yaml.unlink()
print()

# ── Process each enabled source ───────────────────────────────
print("📦 Assembling enabled sources:")
print("=" * 65)

_enabled_srcs = {k: v for k, v in DATASET_SOURCES.items() if v["enabled"]}

if not _enabled_srcs:
    print("⚠️  No sources are enabled — set enabled=True in the Configuration cell.")
else:
    for src_name, src_cfg in _enabled_srcs.items():
        _src_root = (_nb_dir / src_cfg["input_root"]).resolve()
        _map_file = (_nb_dir / src_cfg["class_map"]).resolve()

        print(f"▶  {src_name}")
        print(f"   {src_cfg['note']}")

        if not _src_root.exists():
            print(f"   ❌ INPUT NOT FOUND: {_src_root}")
            print()
            continue
        if not _map_file.exists():
            print(f"   ❌ CLASS MAP NOT FOUND: {_map_file}")
            print()
            continue

        _cmd = [
            sys.executable, str(_remap_py),
            "--input-root",   str(_src_root),
            "--output-root",  str(_assembled),
            "--class-map",    str(_map_file),
            "--target-names", TARGET_NAMES,
            "--dataset-tag",  src_name,
            "--split-map",    src_cfg["split_map"],
        ] + src_cfg.get("extra_args", [])

        _proc = subprocess.run(_cmd, capture_output=True, text=True, cwd=str(_root_dir))

        if _proc.returncode != 0:
            print(f"   ❌ FAILED (exit {_proc.returncode})")
            for _line in (_proc.stderr or "").strip().splitlines()[-8:]:
                print(f"      {_line}")
        else:
            for _line in (_proc.stdout or "").strip().splitlines():
                if _line.strip():
                    print(f"   {_line.strip()}")
            print(f"   ✅ OK")
        print()

print("=" * 65)
print("Assembled totals:")
for split in _SPLITS:
    _n = len(list((_assembled / split / "images").glob("*")))
    print(f"   {split:6s} → {_n:5d} images")
print()
print("✅ Assembly complete — proceed to Dataset Verification ↓")


---
### 4 · Dataset Verification

Verify the assembled dataset structure, class counts, and annotation quality before training.


In [ ]:
# Load and display data.yaml
print("\U0001f50d Loading dataset configuration...")
print()

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

NUM_CLASSES = data_cfg["nc"]
CLASS_NAMES = [data_cfg["names"][i] for i in range(NUM_CLASSES)]

print(f"\u2705 Dataset config loaded: {DATA_YAML}")
print(f"   Number of classes : {NUM_CLASSES}")
print(f"   Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(f"     {i}: {name}")
print()
print(f"   Train images : {data_cfg.get('train', 'N/A')}")
print(f"   Val images   : {data_cfg.get('val', 'N/A')}")
print(f"   Test images  : {data_cfg.get('test', 'N/A')}")

In [ ]:
# Verify dataset split counts & annotation integrity
print("📊 DATASET SUMMARY")
print("=" * 70)

# Resolve the dataset root from data.yaml's 'path' field.
# Convention: 'path' is relative to the YAML file location.
_yaml_path_val = data_cfg.get("path", ".")
if os.path.isabs(_yaml_path_val):
    dataset_root = Path(_yaml_path_val)
else:
    _yaml_dir = Path(DATA_YAML).resolve().parent
    dataset_root = (_yaml_dir / _yaml_path_val).resolve()
dataset_root = str(dataset_root)

splits = ["train", "val", "test"]
split_stats = {}

for split in splits:
    img_dir   = os.path.join(dataset_root, split, "images")
    label_dir = os.path.join(dataset_root, split, "labels")

    if os.path.exists(img_dir):
        images = [f for f in os.listdir(img_dir)
                  if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]
    else:
        images = []

    if os.path.exists(label_dir):
        labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]
    else:
        labels = []

    split_stats[split] = {"images": len(images), "labels": len(labels)}
    status = "✅" if len(images) > 0 and len(images) == len(labels) else "⚠️"
    print(f"   {status} {split:6s} — {len(images):5d} images, {len(labels):5d} labels")

total_images = sum(s["images"] for s in split_stats.values())
total_labels = sum(s["labels"] for s in split_stats.values())

print("=" * 70)
print(f"   Total: {total_images} images, {total_labels} labels")
print(f"   Dataset root: {dataset_root}")

if total_images == 0:
    print()
    print("⚠️  No images found! Please assemble the selected sources into data/annotated/")
    print("   Expected structure:")
    print("     data/annotated/train/images/  &  data/annotated/train/labels/")
    print("     data/annotated/val/images/    &  data/annotated/val/labels/")
    print("     data/annotated/test/images/   &  data/annotated/test/labels/")


In [ ]:
# Per-class annotation distribution (across all splits)
print("\U0001f4ca PER-CLASS ANNOTATION DISTRIBUTION")
print("=" * 70)

class_counts = Counter()

for split in splits:
    label_dir = os.path.join(dataset_root, split, "labels")
    if not os.path.exists(label_dir):
        continue
    for label_file in os.listdir(label_dir):
        if not label_file.endswith(".txt"):
            continue
        filepath = os.path.join(label_dir, label_file)
        with open(filepath, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:  # class_id + 4 bbox coords minimum
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1

if class_counts:
    print(f"{'Class ID':<10} {'Class Name':<30} {'Annotations':>12}")
    print("-" * 55)
    for i in range(NUM_CLASSES):
        count = class_counts.get(i, 0)
        print(f"{i:<10} {CLASS_NAMES[i]:<30} {count:>12}")
    print("-" * 55)
    print(f"{'Total':<41} {sum(class_counts.values()):>12}")
else:
    print("   No annotations found — dataset is empty or not yet exported.")

In [ ]:
# Visualize class distribution (bar chart)
if class_counts:
    fig, ax = plt.subplots(figsize=(12, 5))

    counts = [class_counts.get(i, 0) for i in range(NUM_CLASSES)]
    colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

    bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor="black", linewidth=0.5)

    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(count), ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_title("MosKita — Per-Class Annotation Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("Class", fontsize=11)
    ax.set_ylabel("Number of Annotations", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Skipping visualization — no annotations found.")

In [ ]:

# Per-split class distribution — grouped bar chart + balance check
print("📊 PER-SPLIT CLASS DISTRIBUTION")
print("=" * 70)

from collections import Counter as _Counter

split_counts = {}
for split in splits:
    lbl_dir = os.path.join(dataset_root, split, "labels")
    cnt = _Counter()
    if os.path.exists(lbl_dir):
        for lf in os.listdir(lbl_dir):
            if lf.endswith(".txt"):
                with open(os.path.join(lbl_dir, lf)) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) >= 5:
                            cnt[int(parts[0])] += 1
    split_counts[split] = cnt

# Table
print(f"{'Class':<28}", end="")
for s in splits:
    print(f" {s:>9}", end="")
print(f"  {'Total':>9}")
print("-" * 70)
for i in range(NUM_CLASSES):
    row_total = sum(split_counts[s].get(i, 0) for s in splits)
    print(f"  {CLASS_NAMES[i]:<26}", end="")
    for s in splits:
        print(f" {split_counts[s].get(i, 0):>9}", end="")
    print(f"  {row_total:>9}")
print("-" * 70)

# Grouped bar chart
x      = np.arange(NUM_CLASSES)
width  = 0.26
_clrs  = {"train": "#4C72B0", "val": "#DD8452", "test": "#55A868"}

fig, ax = plt.subplots(figsize=(14, 6))
for idx, split in enumerate(splits):
    counts = [split_counts[split].get(i, 0) for i in range(NUM_CLASSES)]
    ax.bar(x + (idx - 1) * width, counts, width,
           label=split, color=_clrs.get(split, "#888888"),
           edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
ax.set_title("MosKita — Class Distribution per Split", fontsize=14, fontweight="bold")
ax.set_xlabel("Class", fontsize=11)
ax.set_ylabel("Annotation Count", fontsize=11)
ax.legend(title="Split", fontsize=10)
plt.tight_layout()
plt.show()

# Balance check (overall)
if class_counts:
    _total = sum(class_counts.values())
    print("\n⚖️  Overall balance (all splits combined):")
    for i in range(NUM_CLASSES):
        pct = class_counts.get(i, 0) / _total * 100
        bar = "█" * int(pct / 2)
        flag = "  ⚠️  under-represented" if pct < 5 else ""
        print(f"   {CLASS_NAMES[i]:<28} {pct:5.1f}%  {bar}{flag}")


In [ ]:

# Bounding box size, aspect ratio, and annotation density analysis
print("📐 BOUNDING BOX & ANNOTATION ANALYSIS")
print("=" * 70)

all_widths, all_heights, all_areas, all_ratios, imgs_ann_count = [], [], [], [], []

for split in splits:
    lbl_dir = os.path.join(dataset_root, split, "labels")
    if not os.path.exists(lbl_dir):
        continue
    for lf in os.listdir(lbl_dir):
        if not lf.endswith(".txt"):
            continue
        rows = []
        with open(os.path.join(lbl_dir, lf)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    rows.append(parts)
        imgs_ann_count.append(len(rows))
        for parts in rows:
            w, h = float(parts[3]), float(parts[4])
            all_widths.append(w)
            all_heights.append(h)
            all_areas.append(w * h)
            all_ratios.append(w / h if h > 0 else 0)

if all_areas:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1 — Box area distribution
    ax = axes[0, 0]
    ax.hist(all_areas, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax.axvline(float(np.median(all_areas)), color="red",    linestyle="--",
               label=f"Median: {np.median(all_areas):.5f}")
    ax.axvline(float(np.mean(all_areas)),   color="orange", linestyle="--",
               label=f"Mean:   {np.mean(all_areas):.5f}")
    # COCO-style size thresholds (normalised area equivalents at 640px)
    for thresh, lbl in [(0.0032, "small|med"), (0.04, "med|large")]:
        ax.axvline(thresh, color="#AAAAAA", linestyle=":", linewidth=1.2)
        ax.text(thresh, ax.get_ylim()[1] * 0.92, lbl, fontsize=7, color="#888888", ha="left")
    ax.set_title("Bounding Box Area (normalised w×h)", fontweight="bold")
    ax.set_xlabel("Normalised area")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

    # 2 — Aspect ratio distribution
    ax = axes[0, 1]
    clipped = np.clip(all_ratios, 0.05, 8)
    ax.hist(clipped, bins=60, color="#55A868", edgecolor="white", linewidth=0.4)
    ax.axvline(1.0, color="red",    linestyle="--", label="Square (1:1)")
    ax.axvline(float(np.median(clipped)), color="orange", linestyle="--",
               label=f"Median: {np.median(clipped):.2f}")
    ax.set_title("Bounding Box Aspect Ratio (w / h)", fontweight="bold")
    ax.set_xlabel("Width / Height")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

    # 3 — Width vs Height scatter
    ax = axes[1, 0]
    _n = min(5000, len(all_widths))
    ax.scatter(all_widths[:_n], all_heights[:_n], alpha=0.18, s=7, c="#DD8452")
    ax.plot([0, 1], [0, 1], "r--", linewidth=1, label="Square diagonal")
    ax.set_title(f"Box Width vs Height (first {_n:,}, normalised)", fontweight="bold")
    ax.set_xlabel("Width (normalised)")
    ax.set_ylabel("Height (normalised)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

    # 4 — Annotations per image
    ax = axes[1, 1]
    _max_ann = max(imgs_ann_count)
    ax.hist(imgs_ann_count, bins=range(0, _max_ann + 2), color="#C44E52",
            edgecolor="white", linewidth=0.4)
    ax.axvline(float(np.mean(imgs_ann_count)), color="orange", linestyle="--",
               label=f"Mean: {np.mean(imgs_ann_count):.1f}")
    ax.set_title("Annotations per Image", fontweight="bold")
    ax.set_xlabel("Annotations")
    ax.set_ylabel("Images")
    ax.legend(fontsize=8)

    plt.suptitle("MosKita — Bounding Box & Annotation Analysis",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # Summary stats
    _total = len(all_areas)
    _small  = sum(1 for a in all_areas if a < 0.0032)
    _medium = sum(1 for a in all_areas if 0.0032 <= a < 0.04)
    _large  = sum(1 for a in all_areas if a >= 0.04)
    print(f"\n📏 Object size breakdown (COCO-style thresholds on normalised area):")
    print(f"   Small  (< ~32² px equiv.)  : {_small:5d}  ({_small  / _total * 100:.1f}%)")
    print(f"   Medium (~32²–128² px range): {_medium:5d}  ({_medium / _total * 100:.1f}%)")
    print(f"   Large  (> ~128² px equiv.) : {_large:5d}  ({_large  / _total * 100:.1f}%)")
    print(f"\n   Total annotations         : {_total:,}")
    print(f"   Median box area           : {np.median(all_areas):.5f}")
    print(f"   Mean aspect ratio (w/h)   : {np.mean(all_ratios):.3f}")
    print(f"   Mean annotations / image  : {np.mean(imgs_ann_count):.2f}  "
          f"(max: {max(imgs_ann_count)})")
else:
    print("⚠️  No annotations found — run Dataset Assembly first.")


In [ ]:

# Sample training images with ground-truth bounding boxes overlaid
import matplotlib.patches as mpatches

train_img_dir   = os.path.join(dataset_root, "train", "images")
train_label_dir = os.path.join(dataset_root, "train", "labels")

if os.path.exists(train_img_dir):
    _img_files = [f for f in os.listdir(train_img_dir)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if _img_files:
        num_samples = min(8, len(_img_files))
        selected    = random.sample(_img_files, num_samples)
        _cmap       = plt.get_cmap("tab10", NUM_CLASSES)

        cols = 4
        rows = (num_samples + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
        axes = axes.flatten() if num_samples > 1 else [axes]

        for idx, img_name in enumerate(selected):
            img_path = os.path.join(train_img_dir, img_name)
            lbl_path = os.path.join(train_label_dir,
                                    os.path.splitext(img_name)[0] + ".txt")
            img = mpimg.imread(img_path)
            ih, iw = img.shape[:2]
            axes[idx].imshow(img)

            if os.path.exists(lbl_path):
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5:
                            continue
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = (cx - bw / 2) * iw
                        y1 = (cy - bh / 2) * ih
                        pw, ph = bw * iw, bh * ih
                        color = _cmap(cls_id % NUM_CLASSES)
                        rect = mpatches.FancyBboxPatch(
                            (x1, y1), pw, ph,
                            boxstyle="square,pad=0",
                            linewidth=1.5, edgecolor=color, facecolor="none",
                        )
                        axes[idx].add_patch(rect)
                        label = CLASS_NAMES[cls_id] if cls_id < NUM_CLASSES else str(cls_id)
                        axes[idx].text(
                            x1, max(y1 - 3, 0), label,
                            fontsize=6, color=color, fontweight="bold",
                            bbox=dict(boxstyle="square,pad=0.1",
                                      facecolor="black", alpha=0.55, linewidth=0),
                        )

            axes[idx].set_title(img_name[:35], fontsize=7)
            axes[idx].axis("off")

        for idx in range(num_samples, len(axes)):
            axes[idx].axis("off")

        # Class legend
        _handles = [mpatches.Patch(color=_cmap(i), label=CLASS_NAMES[i])
                    for i in range(NUM_CLASSES)]
        fig.legend(_handles, [h.get_label() for h in _handles],
                   loc="lower center", ncol=4, fontsize=9,
                   title="Classes", bbox_to_anchor=(0.5, -0.02))
        plt.suptitle("Sample Training Images — Ground-Truth Boxes",
                     fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  No images in train/images/ — run Dataset Assembly first.")
else:
    print("⚠️  Train images directory not found.")


---
### 5 · Model Training

Fine-tune YOLOv8s on the assembled MosKita dataset. Weights, plots, and metrics are saved to `models/runs/`.


In [ ]:
# Load pretrained YOLOv8s
print("\U0001f680 Loading pretrained model...")
model = YOLO(MODEL_VARIANT)
print(f"\u2705 Model loaded: {MODEL_VARIANT}")
print(f"   Architecture : YOLOv8s (small)")
print(f"   Task         : Object Detection")

In [ ]:
# Train
print("\U0001f3cb\ufe0f  Starting training...")
print("=" * 50)

_t_start = time.perf_counter()

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=RUN_NAME,
    project=PROJECT_DIR,
    exist_ok=EXIST_OK,
    patience=PATIENCE,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_momentum=WARMUP_MOMENTUM,
    close_mosaic=CLOSE_MOSAIC,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    flipud=FLIPUD,
    fliplr=FLIPLR,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    perspective=PERSPECTIVE,
    cache=CACHE,
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    save_period=SAVE_PERIOD,
    plots=True,
    verbose=True,
)

_elapsed = time.perf_counter() - _t_start
_h, _rem = divmod(int(_elapsed), 3600)
_m, _s   = divmod(_rem, 60)

print()
print("\u2705 Training complete!")
print(f"   Elapsed time : {_h:02d}h {_m:02d}m {_s:02d}s")
if torch.cuda.is_available():
    print(f"   Peak VRAM    : {torch.cuda.max_memory_allocated(0) / (1024**3):.2f} GB")


---
### 6 · Results & Metrics

Display training curves, loss components, per-class performance, confusion matrix, and PR curves.


In [ ]:
# Locate the training run output directory
run_dir = os.path.join(PROJECT_DIR, RUN_NAME)

# If run was auto-incremented (e.g. moskita_v12), find the latest
if not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(PROJECT_DIR, f"{RUN_NAME}*")))
    run_dir = candidates[-1] if candidates else run_dir

print(f"\U0001f4c2 Training run directory: {run_dir}")
print()

# List available output files
if os.path.exists(run_dir):
    for item in sorted(os.listdir(run_dir)):
        print(f"   {item}")
else:
    print("\u26a0\ufe0f  Run directory not found — training may not have completed.")

In [ ]:
# Display training results summary
results_csv = os.path.join(run_dir, "results.csv")

if os.path.exists(results_csv):
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    _map50_col     = "metrics/mAP50(B)"
    _map5095_col   = "metrics/mAP50-95(B)"
    _precision_col = "metrics/precision(B)"
    _recall_col    = "metrics/recall(B)"

    _missing = [c for c in [_map50_col, _map5095_col, _precision_col, _recall_col]
                if c not in df.columns]
    if _missing:
        print(f"\u26a0\ufe0f  Unexpected CSV columns. Available: {list(df.columns)}")
    else:
        best_idx = df[_map50_col].idxmax()
        best_row = df.iloc[best_idx]

        print("\U0001f3c6 TRAINING RESULTS SUMMARY")
        print("=" * 50)
        print(f"   Total epochs      : {len(df)}")
        print(f"   Best epoch        : {int(best_row['epoch']) + 1}")
        print(f"   Best mAP@50       : {best_row[_map50_col]:.4f}")
        print(f"   Best mAP@50-95    : {best_row[_map5095_col]:.4f}")
        print(f"   Best Precision    : {best_row[_precision_col]:.4f}")
        print(f"   Best Recall       : {best_row[_recall_col]:.4f}")

        # Box loss at best epoch
        for loss_col in ["train/box_loss", "val/box_loss"]:
            if loss_col in df.columns:
                print(f"   {loss_col:<22}: {best_row[loss_col]:.4f}")
        print("=" * 50)

        # Target assessment
        map50 = best_row[_map50_col]
        if map50 > 0.85:
            level = "\U0001f31f PUBLISHABLE"
        elif map50 > 0.75:
            level = "\u2705 GOOD"
        elif map50 > 0.60:
            level = "\U0001f44d ACCEPTABLE"
        else:
            level = "\u26a0\ufe0f  BELOW TARGET — consider more data or tuning"
        print(f"   Performance level : {level}")
else:
    print("\u26a0\ufe0f  results.csv not found — training may not have completed.")


In [ ]:

# Per-class AP — run model.val() on the best checkpoint
_bw = os.path.join(run_dir, "weights", "best.pt")
if not os.path.exists(_bw):
    _bw = os.path.join(run_dir, "weights", "last.pt")

if os.path.exists(_bw):
    print("🔬 Running per-class validation on best weights...")
    _eval_model  = YOLO(_bw)
    _val_results = _eval_model.val(
        data=DATA_YAML, imgsz=IMG_SIZE,
        conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
        verbose=False,
    )

    if hasattr(_val_results, "box"):
        _box = _val_results.box

        # ── Table ─────────────────────────────────────────────
        print(f"\n{'Class':<28} {'AP@50':>7} {'AP@50-95':>9} {'Precision':>10} {'Recall':>8}")
        print("-" * 67)
        for i, cls_name in enumerate(CLASS_NAMES):
            _ap50   = float(_box.ap50[i])   if hasattr(_box, "ap50") else float("nan")
            _ap5095 = float(_box.ap[i])     if hasattr(_box, "ap")   else float("nan")
            _prec   = float(_box.p[i])      if hasattr(_box, "p")    else float("nan")
            _rec    = float(_box.r[i])      if hasattr(_box, "r")    else float("nan")
            print(f"  {cls_name:<26} {_ap50:>7.4f} {_ap5095:>9.4f} {_prec:>10.4f} {_rec:>8.4f}")
        print("-" * 67)
        print(f"  {'Overall (mean)':<26} {_box.map50:>7.4f} {_box.map:>9.4f}")

        # ── Per-class AP@50 bar chart ──────────────────────────
        _ap50_vals = [float(_box.ap50[i]) for i in range(len(CLASS_NAMES))]
        _norm = plt.Normalize(vmin=0, vmax=1)
        _bar_colors = plt.cm.RdYlGn(_norm(_ap50_vals))

        fig, ax = plt.subplots(figsize=(12, 5))
        _bars = ax.bar(CLASS_NAMES, _ap50_vals,
                       color=_bar_colors, edgecolor="black", linewidth=0.5)
        for bar, val in zip(_bars, _ap50_vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.01,
                    f"{val:.3f}", ha="center", va="bottom",
                    fontsize=9, fontweight="bold")

        for threshold, color, label in [
            (0.60, "#FFA500", "Acceptable (0.60)"),
            (0.75, "#2196F3", "Good (0.75)"),
            (0.85, "#4CAF50", "Publishable (0.85)"),
        ]:
            ax.axhline(threshold, color=color, linestyle="--", linewidth=1.2, label=label)

        ax.set_ylim(0, 1.1)
        ax.set_title("Per-Class AP@50", fontsize=14, fontweight="bold")
        ax.set_xlabel("Class", fontsize=11)
        ax.set_ylabel("AP@50", fontsize=11)
        ax.legend(fontsize=9)
        plt.xticks(rotation=45, ha="right", fontsize=9)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Val results did not expose .box metrics — check Ultralytics version.")
else:
    print("⚠️  No weights found in run directory — run training first.")


In [ ]:
# Training curves
results_png = os.path.join(run_dir, "results.png")

if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  results.png not found.")

In [ ]:

# Training loss components over epochs (box / class / DFL)
if os.path.exists(results_csv):
    import pandas as pd
    _df = pd.read_csv(results_csv)
    _df.columns = _df.columns.str.strip()

    _loss_pairs = [
        ("Box Loss",   "train/box_loss",  "val/box_loss"),
        ("Class Loss", "train/cls_loss",  "val/cls_loss"),
        ("DFL Loss",   "train/dfl_loss",  "val/dfl_loss"),
    ]
    _avail = [(name, tc, vc) for name, tc, vc in _loss_pairs
              if tc in _df.columns and vc in _df.columns]

    if _avail:
        _epochs_col = _df["epoch"] if "epoch" in _df.columns else range(len(_df))
        fig, axes = plt.subplots(1, len(_avail), figsize=(6 * len(_avail), 5))
        if len(_avail) == 1:
            axes = [axes]

        for ax, (name, tc, vc) in zip(axes, _avail):
            ax.plot(_epochs_col, _df[tc], label="train", color="#4C72B0", linewidth=1.5)
            ax.plot(_epochs_col, _df[vc], label="val",   color="#DD8452", linewidth=1.5)
            ax.set_title(name, fontweight="bold")
            ax.set_xlabel("Epoch")
            ax.set_ylabel("Loss")
            ax.legend(fontsize=9)
            ax.grid(alpha=0.25)

        plt.suptitle("Training Loss Components", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Loss component columns not found in results.csv.")
        print(f"   Available columns: {list(_df.columns)}")
else:
    print("⚠️  results.csv not found — training may not have completed.")


In [ ]:
# Confusion matrix
confusion_png = os.path.join(run_dir, "confusion_matrix.png")
confusion_norm_png = os.path.join(run_dir, "confusion_matrix_normalized.png")

plot_files = []
if os.path.exists(confusion_png):
    plot_files.append(("Confusion Matrix", confusion_png))
if os.path.exists(confusion_norm_png):
    plot_files.append(("Confusion Matrix (Normalized)", confusion_norm_png))

if plot_files:
    fig, axes = plt.subplots(1, len(plot_files), figsize=(10 * len(plot_files), 10))
    if len(plot_files) == 1:
        axes = [axes]

    for ax, (title, path) in zip(axes, plot_files):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Confusion matrix plots not found.")

In [ ]:
# Precision-Recall and F1 curves
pr_plots = [
    ("Precision-Recall Curve", os.path.join(run_dir, "PR_curve.png")),
    ("F1 Score Curve", os.path.join(run_dir, "F1_curve.png")),
    ("Precision Curve", os.path.join(run_dir, "P_curve.png")),
    ("Recall Curve", os.path.join(run_dir, "R_curve.png")),
]

available = [(t, p) for t, p in pr_plots if os.path.exists(p)]

if available:
    cols = min(2, len(available))
    rows = (len(available) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 8 * rows))
    axes = np.array(axes).flatten() if len(available) > 1 else [axes]

    for idx, (title, path) in enumerate(available):
        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].axis("off")
        axes[idx].set_title(title, fontsize=13, fontweight="bold")

    for idx in range(len(available), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  PR / F1 curve plots not found.")

In [ ]:
# Validation batch predictions (visual check)
val_preds = [
    ("Val Batch 0 — Labels", os.path.join(run_dir, "val_batch0_labels.jpg")),
    ("Val Batch 0 — Predictions", os.path.join(run_dir, "val_batch0_pred.jpg")),
]

available_val = [(t, p) for t, p in val_preds if os.path.exists(p)]

if available_val:
    fig, axes = plt.subplots(1, len(available_val), figsize=(12 * len(available_val), 10))
    if len(available_val) == 1:
        axes = [axes]

    for ax, (title, path) in zip(axes, available_val):
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(title, fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Validation prediction images not found.")

---
### 7 · Model Export (ONNX / TFLite)

Export the best weights for Raspberry Pi 5 edge deployment.


In [ ]:
# Load best weights from the training run
best_weights = os.path.join(run_dir, "weights", "best.pt")

if os.path.exists(best_weights):
    print(f"\U0001f4e6 Loading best weights: {best_weights}")
    best_model = YOLO(best_weights)
    print("\u2705 Best model loaded successfully!")
else:
    print("\u26a0\ufe0f  best.pt not found — using last trained model instead.")
    last_weights = os.path.join(run_dir, "weights", "last.pt")
    if os.path.exists(last_weights):
        best_model = YOLO(last_weights)
        print(f"\u2705 Last model loaded: {last_weights}")
    else:
        best_model = model
        print("\u26a0\ufe0f  No saved weights found — using current model in memory.")

In [ ]:
import shutil

# Export to ONNX (dynamic batch axis, simplified graph)
print("\U0001f680 Exporting to ONNX...")
onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    simplify=True,
    dynamic=True,        # enables variable batch size at inference time
)
print(f"\u2705 ONNX exported: {onnx_path}")

# Copy to exports directory
os.makedirs(EXPORT_DIR, exist_ok=True)
onnx_dest = os.path.join(EXPORT_DIR, "moskita.onnx")
shutil.copy2(onnx_path, onnx_dest)
print(f"\u2705 Copied to: {onnx_dest}")


In [ ]:
# Export to TFLite (for Pi 5 lightweight inference)
print("\U0001f680 Exporting to TFLite...")
try:
    tflite_path = best_model.export(format="tflite", imgsz=IMG_SIZE)
    print(f"\u2705 TFLite exported: {tflite_path}")

    tflite_dest = os.path.join(EXPORT_DIR, "moskita.tflite")
    shutil.copy2(tflite_path, tflite_dest)
    print(f"\u2705 Copied to: {tflite_dest}")
except Exception as e:
    print(f"\u26a0\ufe0f  TFLite export failed: {e}")
    print("   TFLite export requires tensorflow. Install with: pip install tensorflow")
    print("   You can export on the Pi 5 itself or install TF on this machine.")

---
### 8 · Quick Inference Test

Run a quick inference on a sample image to verify the trained model works end-to-end.


In [ ]:
# Find a sample test image
test_img_dir = os.path.join(dataset_root, "test", "images")
val_img_dir  = os.path.join(dataset_root, "val", "images")

sample_path = None
for search_dir in [test_img_dir, val_img_dir, train_img_dir]:
    if os.path.exists(search_dir):
        candidates = [f for f in os.listdir(search_dir)
                      if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if candidates:
            sample_path = os.path.join(search_dir, random.choice(candidates))
            break

if sample_path:
    print(f"\U0001f50e Running inference on: {os.path.basename(sample_path)}")
    print()

    inf_results = best_model(sample_path, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD)

    # Display results
    for r in inf_results:
        annotated = r.plot()  # BGR numpy array with bounding boxes
        annotated_rgb = annotated[:, :, ::-1]  # convert BGR → RGB for matplotlib

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(annotated_rgb)
        ax.axis("off")
        ax.set_title("MosKita — Sample Detection", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

        # Print detection details
        # Any detected object is a potential breeding site by definition.
        if r.boxes is not None and len(r.boxes) > 0:
            print(f"\U0001f4cb Detections: {len(r.boxes)}")
            print(f"{'Class':<30} {'Confidence':>10}")
            print("-" * 43)
            for box in r.boxes:
                cls_name = CLASS_NAMES[int(box.cls)]
                conf = float(box.conf)
                print(f"{cls_name:<30} {conf:>10.4f}")
        else:
            print("No detections found in this image.")
else:
    print("\u26a0\ufe0f  No sample images available for inference test.")
    print("   Add images to your dataset first, then re-run this cell.")


---
### 9 · Training Summary & Next Steps

**What was done:**
1. Assembled selected dataset sources and verified class distribution
2. Trained YOLOv8s on the MosKita V1 dataset
3. Evaluated performance with mAP, per-class AP, confusion matrix, and PR curves
4. Exported model to ONNX and TFLite for Pi 5 deployment

**Next steps:**
- Run `evaluation.ipynb` for detailed per-class analysis
- If mAP@50 < 0.75, apply the **active learning loop** (see `Docs/MOSKITA_CONTEXT.md §16`)
- Deploy ONNX/TFLite model to Raspberry Pi 5 with `deploy/pi_inference.py`

**Target Metrics:**

| Metric | Acceptable | Good | Publishable |
|---|---|---|---|
| mAP@50 | >0.60 | >0.75 | >0.85 |
| Precision | >0.65 | >0.78 | >0.88 |
| Recall | >0.60 | >0.75 | >0.83 |
| Inference (Pi 5) | <500 ms | <200 ms | <100 ms |
